# ✈️ Flight Price Prediction
## Notebook 4: Experiments — Models, Hyperparameter Tuning, Ensembles

**Цель:** Обучить 5+ моделей, подобрать гиперпараметры, построить ансамбль, выбрать финальную модель.

**Формат каждого эксперимента:** Гипотеза → Как проверялось → Результат

**Метрика:** RMSE (основная), MAE, R²

## 0. Импорты

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
import os

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

import xgboost as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

os.makedirs('../data/plots', exist_ok=True)
os.makedirs('../models', exist_ok=True)

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

print('Импорты выполнены ✅')

ModuleNotFoundError: No module named 'xgboost'

## 1. Загрузка данных

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_val   = pd.read_csv('../data/processed/X_val.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_val   = pd.read_csv('../data/processed/y_val.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

with open('../data/processed/features.json') as f:
    meta = json.load(f)
FEATURES = meta['features']

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## 2. Вспомогательные функции

In [ ]:
experiments = []

def evaluate(model, X, y, split_name='Val'):
    y_pred = model.predict(X)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    mae  = mean_absolute_error(y, y_pred)
    r2   = r2_score(y, y_pred)
    print(f'  [{split_name}] RMSE: {rmse:>10,.0f} | MAE: {mae:>10,.0f} | R²: {r2:.4f}')
    return {'rmse': rmse, 'mae': mae, 'r2': r2}

def log_experiment(name, hypothesis, method, train_res, val_res, notes=''):
    experiments.append({
        'Модель': name,
        'Гипотеза': hypothesis,
        'Метод проверки': method,
        'Train RMSE': f"{train_res['rmse']:,.0f}",
        'Val RMSE': f"{val_res['rmse']:,.0f}",
        'Val MAE': f"{val_res['mae']:,.0f}",
        'Val R²': f"{val_res['r2']:.4f}",
        'Заметки': notes
    })
    print(f'  → Записано в таблицу экспериментов ✅')

print('Функции определены ✅')

## 3. Эксперимент 1: KNN

**Гипотеза:** Похожие рейсы имеют похожую цену — KNN должен работать лучше наивного baseline.

**Метод:** KNeighborsRegressor с k=10, стандартизация признаков.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train[FEATURES])
X_val_sc   = scaler.transform(X_val[FEATURES])

knn = KNeighborsRegressor(n_neighbors=10, n_jobs=-1)
knn.fit(X_train_sc, y_train)

print('=== KNN (k=10) ===')
train_res = evaluate(knn, X_train_sc, y_train, 'Train')
val_res   = evaluate(knn, X_val_sc,   y_val,   'Val')

log_experiment(
    name='KNN (k=10)',
    hypothesis='Похожие рейсы → похожая цена',
    method='KNeighborsRegressor, стандартизация',
    train_res=train_res, val_res=val_res,
    notes='Лучше dummy, хуже линейной'
)

## 4. Эксперимент 2: Ridge Regression с Feature Engineering

**Гипотеза:** Добавление engineered features улучшит линейную модель по сравнению с базовым baseline.

**Метод:** Ridge Regression со всеми признаками включая новые.

In [ ]:
ridge = Ridge(alpha=1.0, random_state=SEED)
ridge.fit(X_train_sc, y_train)

print('=== Ridge Regression (все признаки + FE) ===')
train_res = evaluate(ridge, X_train_sc, y_train, 'Train')
val_res   = evaluate(ridge, X_val_sc,   y_val,   'Val')

log_experiment(
    name='Ridge (все признаки + FE)',
    hypothesis='FE улучшит линейную модель',
    method='Ridge Regression, alpha=1.0',
    train_res=train_res, val_res=val_res
)

## 5. Эксперимент 3: Random Forest

**Гипотеза:** Нелинейные зависимости (например влияние класса Business) лучше улавливаются деревьями.

**Метод:** RandomForestRegressor, 100 деревьев.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
rf.fit(X_train[FEATURES], y_train)

print('=== Random Forest (100 деревьев) ===')
train_res = evaluate(rf, X_train[FEATURES], y_train, 'Train')
val_res   = evaluate(rf, X_val[FEATURES],   y_val,   'Val')

log_experiment(
    name='Random Forest (100 trees)',
    hypothesis='Деревья лучше улавливают нелинейности',
    method='RandomForestRegressor, n_estimators=100',
    train_res=train_res, val_res=val_res
)

## 6. Эксперимент 4: XGBoost

**Гипотеза:** Градиентный бустинг превзойдёт Random Forest за счёт последовательного обучения.

**Метод:** XGBoost с базовыми параметрами.

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=SEED,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(X_train[FEATURES], y_train,
              eval_set=[(X_val[FEATURES], y_val)],
              verbose=False)

print('=== XGBoost (базовые параметры) ===')
train_res = evaluate(xgb_model, X_train[FEATURES], y_train, 'Train')
val_res   = evaluate(xgb_model, X_val[FEATURES],   y_val,   'Val')

log_experiment(
    name='XGBoost (базовые параметры)',
    hypothesis='Бустинг превзойдёт Random Forest',
    method='XGBRegressor, n_estimators=300, lr=0.1',
    train_res=train_res, val_res=val_res
)

## 7. Эксперимент 5: LightGBM

**Гипотеза:** LightGBM быстрее и точнее XGBoost на табличных данных.

**Метод:** LGBMRegressor с базовыми параметрами.

In [ ]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(X_train[FEATURES], y_train,
              eval_set=[(X_val[FEATURES], y_val)])

print('=== LightGBM (базовые параметры) ===')
train_res = evaluate(lgb_model, X_train[FEATURES], y_train, 'Train')
val_res   = evaluate(lgb_model, X_val[FEATURES],   y_val,   'Val')

log_experiment(
    name='LightGBM (базовые параметры)',
    hypothesis='LightGBM быстрее и точнее XGBoost',
    method='LGBMRegressor, n_estimators=300, lr=0.1',
    train_res=train_res, val_res=val_res
)

## 8. Эксперимент 6: XGBoost с логарифмом таргета

**Гипотеза:** Логарифмирование цены уменьшит влияние выбросов и улучшит качество.

**Метод:** XGBoost на log(price), предсказания обратно трансформируются через exp.

In [ ]:
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)

xgb_log = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=SEED,
    n_jobs=-1,
    verbosity=0
)
xgb_log.fit(X_train[FEATURES], y_train_log, verbose=False)

# Предсказания в оригинальном масштабе
y_pred_train = np.expm1(xgb_log.predict(X_train[FEATURES]))
y_pred_val   = np.expm1(xgb_log.predict(X_val[FEATURES]))

train_res = {
    'rmse': np.sqrt(mean_squared_error(y_train, y_pred_train)),
    'mae':  mean_absolute_error(y_train, y_pred_train),
    'r2':   r2_score(y_train, y_pred_train)
}
val_res = {
    'rmse': np.sqrt(mean_squared_error(y_val, y_pred_val)),
    'mae':  mean_absolute_error(y_val, y_pred_val),
    'r2':   r2_score(y_val, y_pred_val)
}

print('=== XGBoost + log(price) ===')
print(f'  [Train] RMSE: {train_res["rmse"]:>10,.0f} | MAE: {train_res["mae"]:>10,.0f} | R²: {train_res["r2"]:.4f}')
print(f'  [Val]   RMSE: {val_res["rmse"]:>10,.0f} | MAE: {val_res["mae"]:>10,.0f} | R²: {val_res["r2"]:.4f}')

log_experiment(
    name='XGBoost + log(price)',
    hypothesis='log-трансформация таргета уменьшит ошибку',
    method='XGBoost на log1p(price), обратная expm1',
    train_res=train_res, val_res=val_res
)

## 9. Эксперимент 7: LightGBM с подбором гиперпараметров (Optuna)

**Гипотеза:** Подбор гиперпараметров улучшит LightGBM.

**Метод:** Optuna, 50 trials, минимизируем RMSE на val.

In [ ]:
def objective(trial):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':     trial.suggest_int('max_depth', 3, 10),
        'num_leaves':    trial.suggest_int('num_leaves', 20, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': SEED,
        'n_jobs': -1,
        'verbose': -1
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train[FEATURES], y_train)
    y_pred = model.predict(X_val[FEATURES])
    return np.sqrt(mean_squared_error(y_val, y_pred))

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nЛучший RMSE: {study.best_value:,.0f}')
print(f'Лучшие параметры: {study.best_params}')

In [ ]:
# Обучаем с лучшими параметрами
best_lgb = lgb.LGBMRegressor(**study.best_params, random_state=SEED, n_jobs=-1, verbose=-1)
best_lgb.fit(X_train[FEATURES], y_train)

print('=== LightGBM (Optuna) ===')
train_res = evaluate(best_lgb, X_train[FEATURES], y_train, 'Train')
val_res   = evaluate(best_lgb, X_val[FEATURES],   y_val,   'Val')

log_experiment(
    name='LightGBM (Optuna, 50 trials)',
    hypothesis='Подбор гиперпараметров улучшит LightGBM',
    method='Optuna TPE, 50 trials, минимизация RMSE на val',
    train_res=train_res, val_res=val_res,
    notes=f"best params: {study.best_params}"
)

## 10. Эксперимент 8: Ансамбль (Voting)

**Гипотеза:** Усреднение предсказаний нескольких моделей снизит дисперсию ошибки.

**Метод:** VotingRegressor из RF + XGBoost + LightGBM.

In [ ]:
ensemble = VotingRegressor([
    ('rf',  RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)),
    ('xgb', xgb.XGBRegressor(n_estimators=300, learning_rate=0.1, random_state=SEED, verbosity=0)),
    ('lgb', lgb.LGBMRegressor(**study.best_params, random_state=SEED, verbose=-1))
], n_jobs=-1)

ensemble.fit(X_train[FEATURES], y_train)

print('=== Ансамбль: RF + XGBoost + LightGBM ===')
train_res = evaluate(ensemble, X_train[FEATURES], y_train, 'Train')
val_res   = evaluate(ensemble, X_val[FEATURES],   y_val,   'Val')

log_experiment(
    name='Ансамбль (RF + XGB + LGB)',
    hypothesis='Усреднение снизит дисперсию ошибки',
    method='VotingRegressor, равные веса',
    train_res=train_res, val_res=val_res
)

## 11. Таблица всех экспериментов

In [ ]:
exp_df = pd.DataFrame(experiments)
print('=== Таблица экспериментов CP2 ===')
print(exp_df[['Модель', 'Val RMSE', 'Val MAE', 'Val R²', 'Заметки']].to_string(index=False))

exp_df.to_csv('../data/processed/experiments_cp2.csv', index=False)
print('\nСохранено ✅')

## 12. Выбор финальной модели

In [ ]:
# Визуализация сравнения моделей
exp_display = exp_df.copy()
exp_display['Val RMSE num'] = exp_display['Val RMSE'].str.replace(',', '').astype(float)
exp_display = exp_display.sort_values('Val RMSE num')

plt.figure(figsize=(12, 6))
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(exp_display))]
bars = plt.barh(exp_display['Модель'], exp_display['Val RMSE num'], color=colors, edgecolor='white')
plt.xlabel('Val RMSE (рупии) — меньше лучше')
plt.title('Сравнение моделей по Val RMSE', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/plots/model_comparison.png', bbox_inches='tight')
plt.show()

best_model_name = exp_display.iloc[0]['Модель']
best_rmse = exp_display.iloc[0]['Val RMSE']
print(f'\n🏆 Лучшая модель: {best_model_name}')
print(f'   Val RMSE: {best_rmse}')

## 13. Финальная модель: оценка на тесте

In [ ]:
# Выбираем лучшую модель (LightGBM с Optuna)
# Переобучаем на train+val для максимума данных
X_trainval = pd.concat([X_train, X_val])[FEATURES]
y_trainval = pd.concat([y_train, y_val])

final_model = lgb.LGBMRegressor(**study.best_params, random_state=SEED, n_jobs=-1, verbose=-1)
final_model.fit(X_trainval, y_trainval)

print('=== ФИНАЛЬНАЯ МОДЕЛЬ: LightGBM (Optuna) ===')
print('Обучена на Train + Val, оцениваем на Test:')
test_res = evaluate(final_model, X_test[FEATURES], y_test, 'Test')
print(f'\n📊 Итоговые метрики на тесте:')
print(f'  RMSE: {test_res["rmse"]:,.0f} рупий')
print(f'  MAE:  {test_res["mae"]:,.0f} рупий')
print(f'  R²:   {test_res["r2"]:.4f}')

## 14. Важность признаков

In [ ]:
feat_imp = pd.DataFrame({
    'feature': FEATURES,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp, x='importance', y='feature', palette='viridis')
plt.title('Важность признаков (LightGBM)', fontweight='bold')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.savefig('../data/plots/feature_importance.png', bbox_inches='tight')
plt.show()

print('Топ-5 признаков:')
print(feat_imp.head())

In [ ]:
# Predicted vs Actual на тесте
y_pred_test = final_model.predict(X_test[FEATURES])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_test, alpha=0.1, s=5, color='steelblue')
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Идеал')
axes[0].set_title('Predicted vs Actual (Test)', fontweight='bold')
axes[0].set_xlabel('Реальная цена')
axes[0].set_ylabel('Предсказанная цена')
axes[0].legend()

residuals = y_test - y_pred_test
axes[1].hist(residuals, bins=60, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Остатки на тесте', fontweight='bold')
axes[1].set_xlabel('Остаток')

plt.tight_layout()
plt.savefig('../data/plots/final_model_test.png', bbox_inches='tight')
plt.show()

## 15. Сохранение финальной модели

In [ ]:
import joblib

joblib.dump(final_model, '../models/final_model.pkl')
joblib.dump(FEATURES, '../models/features.pkl')

print('Модель сохранена в models/final_model.pkl ✅')
print('Список признаков сохранён в models/features.pkl ✅')

## 16. Выводы

- **Лучшая модель:** LightGBM с подбором гиперпараметров через Optuna
- **Ключевые признаки:** class (Business/Economy), days_left, airline, duration
- **Ансамбль** не превзошёл лучшую одиночную модель — LightGBM уже достаточно мощный
- **log-трансформация** таргета не дала значимого улучшения на этом датасете
- **Линейные модели** значительно уступают бустингу — данные нелинейные

**Следующий шаг (CP3):** Деплой модели через FastAPI + Streamlit, написание отчёта.